# Systemy Big Data: Wykład 1
## Składowanie Masowe: Apache Parquet & Apache Iceberg

Na dzisiejszym laboratorium udowodnimy w praktyce dwa fundamentalne koncepty nowoczesnej Inżynierii Danych:
1. Dlaczego format kolumnowy (**Parquet**) deklasuje formaty tekstowe (**CSV**).
2. Jak **Apache Iceberg** wprowadza transakcje **ACID** i **Time Travel** do zwykłych plików na dysku.

Poniżej inicjalizujemy lokalny klaster Apache Spark i konfigurujemy go tak, aby natywnie obsługiwał format Iceberg.

In [ ]:
import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

# KLUCZOWE: Zabijamy starą, zablokowaną sesję, żeby nowa konfiguracja weszła w życie
try:
    spark.stop()
except:
    pass

print("Konfiguruję bezpieczny, lokalny system plików...")

# Tworzymy czystą sesję. Zwróć uwagę na "defaultCatalog"!
spark = SparkSession.builder \
    .appName("IcebergDemo") \
    .config("spark.sql.catalog.mojabaza", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.mojabaza.type", "hadoop") \
    .config("spark.sql.catalog.mojabaza.warehouse", "/home/iceberg/warehouse") \
    .config("spark.sql.defaultCatalog", "mojabaza") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Środowisko gotowe! Spark uruchomiony poprawnie.")

### 1. Przygotowanie danych testowych

Zanim zaczniemy testować systemy dyskowe, potrzebujemy danych. 
Użyjemy biblioteki `pandas` do wygenerowania w pamięci RAM prostej tabeli zawierającej **3 miliony rekordów** (pracowników fikcyjnej firmy).

In [ ]:
# Generujemy 3 miliony wierszy "na sucho" w Pythonie (Pandas)
print("Generowanie 3 milionów rekordów...")
df_pd = pd.DataFrame({
    'id': np.arange(3000000),
    'name': ['Jan'] * 3000000,
    'city': ['Warszawa'] * 3000000,
    'salary': np.random.randint(4000, 15000, size=3000000)
})
print("Zrobione!")

### 2. Starcie formatów: CSV vs Apache Parquet

Mamy w pamięci 3 miliony wierszy. Zapiszemy je teraz na dysk w dwóch formatach:
* **CSV** - format tekstowy, zorientowany wierszowo (klasyka).
* **Parquet** - format binarny, zorientowany kolumnowo, skompresowany.

Zwróćcie uwagę na miażdżącą różnicę w fizycznym rozmiarze plików na dysku. W systemach Big Data (gdzie płacimy za terabajty w chmurze) to oznacza gigantyczne oszczędności.

In [ ]:
# Zapisujemy jako staroświecki CSV i nowoczesny Parquet
csv_path = '/home/iceberg/warehouse/test.csv'
parquet_path = '/home/iceberg/warehouse/test.parquet'

print("Zapisuję do CSV (to chwilę potrwa)...")
df_pd.to_csv(csv_path, index=False)

print("Zapisuję do Parquet...")
df_pd.to_parquet(parquet_path, index=False)

# Porównujemy rozmiary
csv_size = os.path.getsize(csv_path) / (1024 * 1024)
parquet_size = os.path.getsize(parquet_path) / (1024 * 1024)

print(f"\n--- WYNIKI ---")
print(f"Rozmiar CSV:     {csv_size:.2f} MB")
print(f"Rozmiar Parquet: {parquet_size:.2f} MB")
print(f"Parquet jest {(csv_size/parquet_size):.1f} razy mniejszy!")

### 3. Wejście Data Lakehouse (Apache Iceberg)

Sam plik Parquet jest super, ale... jest **niezmienny (immutable)**. Nie da się w nim łatwo zaktualizować jednego wiersza. 

Rozwiązaniem jest **Table Format**. Podpinamy nasz plik Parquet pod Apache Iceberg. Tworzymy przestrzeń nazw (bazę) i definiujemy prawdziwą tabelę. Od teraz Iceberg zarządza metadanymi tego pliku.

In [ ]:
# Ładujemy Parquet do Sparka
df_spark = spark.read.parquet(parquet_path)

# Tworzymy folder (namespace) wewnątrz naszego katalogu 'mojabaza'
print("Tworzę przestrzeń nazw 'db'...")
spark.sql("CREATE NAMESPACE IF NOT EXISTS mojabaza.db")

print("Tworzę tabelę transakcyjną Iceberg...")
df_spark.writeTo("mojabaza.db.employees").createOrReplace()
print("Tabela gotowa!")

# Pokażmy, ile jest wierszy:
spark.sql("SELECT COUNT(*) as Liczba_Pracownikow FROM mojabaza.db.employees").show()

### 4. Transakcje ACID na "surowych plikach"

W klasycznym Data Lake (np. Hadoop) usunięcie konkretnych wierszy z gigantycznego pliku wymagałoby napisania skryptu, który wczytuje wszystko do pamięci, filtruje i nadpisuje plik. 

Dzięki Icebergowi możemy po prostu użyć klasycznego **SQL DELETE**. Iceberg pod maską zadba o spójność danych i zaktualizuje drzewo metadanych. Zobaczmy to w akcji!

In [ ]:
print("Zwalniamy wszystkich, którzy zarabiają poniżej 10 000 zł!")

spark.sql("DELETE FROM mojabaza.db.employees WHERE salary < 10000")

# Sprawdzamy stan
spark.sql("SELECT COUNT(*) as Liczba_Zatrudnionych_Po_Zwolnieniach FROM mojabaza.db.employees").show()

### 5. Co się stało pod maską? (Snapshots)

Zwolniliśmy ludzi, ale **Iceberg domyślnie nie kasuje od razu starych plików Parquet**. 
Zamiast tego, przy każdej operacji tworzy nową **Migawkę (Snapshot)**. 

Zobaczmy historię operacji na naszej tabeli. Każdy wpis w poniższej tabeli to osobny stan bazy danych w danym ułamku sekundy.

In [ ]:
# Pobieramy historię
snapshots_df = spark.sql("SELECT snapshot_id, committed_at FROM mojabaza.db.employees.snapshots")
snapshots_df.show(truncate=False)

# Bierzemy ID pierwszego snapshota do zmiennej
snapshots_list = snapshots_df.collect()
first_snapshot_id = snapshots_list[0]['snapshot_id']
print(f"ID naszego pierwszego zapisu to: {first_snapshot_id}")

### 6. Time Travel (Podróż w Czasie)

Zarząd firmy zmienił zdanie i cofnął zwolnienia. W relacyjnej bazie bez backupu mielibyśmy ogromny problem. 

W architekturze Lakehouse używamy funkcji **Time Travel**. Wystarczy do zapytania SQL dopisać klauzulę `VERSION AS OF` i podać ID starszego Snapshota. Silnik zignoruje nowe dane i odczyta stare, nienadpisane pliki z dysku.

In [ ]:
print("Szef kazał cofnąć zwolnienia. Używamy Time Travel!")

# Zobacz jak w SQL zapytanie do historii staje się banalne:
query = f"SELECT COUNT(*) as Przywroceni_Pracownicy FROM mojabaza.db.employees VERSION AS OF {first_snapshot_id}"

spark.sql(query).show()
print("Koniec dema! Sukces!")